# Probability of Default (PD) Estimation

### Credit Scoring Risk Assessment — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), familiarity with `scikit-learn` basics, and basic understanding of credit scoring and risk assessment terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the credit scoring and risk assessment problem and why the chosen classification approach suits this banking workflow.
- Implement an end-to-end classification pipeline on synthetic data and evaluate performance with domain-appropriate metrics.
- Assess whether the model is operationally viable, considering error costs, fairness, and the limitations of synthetic data.

**Estimated time:** 35–45 minutes

**Collection context:** Credit scoring, probability of default, loss estimation, and stress testing for banking risk management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** PD is a core component of "Expected Loss" (PD x LGD x EAD). Instead of just saying "this person might default," banks need an exact percentage (e.g., "there is a 2.4% chance this loan will fail next year") to calculate how much capital they must hold in reserve.

**Input data used:** Credit score, loan-to-value (LTV) ratio, debt-to-income (DTI) ratio, and macroeconomic indicators (unemployment rate).

**What we predict:** A continuous probability between 0 and 1 (`pd_estimate`).

**ML method used:** Logistic Regression (outputting probabilities).

**Learning goal:** Understand how to extract and interpret probabilities from classification models.

**Primary stakeholders:** credit analysts, loan officers, risk managers, and regulatory auditors.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Portfolio Dataset

We simulate 10,000 loans. Some are "High Quality" and some are "Subprime."

In [ ]:
n = 10000
rng = RNG

credit_score = rng.normal(700, 80, n).clip(300, 850)
ltv_ratio = rng.uniform(0.4, 1.2, n) # Loan-to-Value
dti_ratio = rng.uniform(0.1, 0.6, n) # Debt-to-Income
unemployment_rate = rng.uniform(0.03, 0.12, n)

# Underlying risk logic
logit = (
    (800 - credit_score) / 100 + 
    3.0 * ltv_ratio + 
    5.0 * dti_ratio + 
    10.0 * unemployment_rate - 
    8.0
)

true_pd = 1 / (1 + np.exp(-logit))
default_event = (rng.random(n) < true_pd).astype(int)

df = pd.DataFrame({
    'credit_score': credit_score,
    'ltv_ratio': ltv_ratio,
    'dti_ratio': dti_ratio,
    'unemployment_rate': unemployment_rate,
    'default_event': default_event
})

print(f"Portfolio Default Rate: {df['default_event'].mean()*100:.2f}%")

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Prepare features for modelling. Domain-meaningful transformations help the model learn patterns aligned with banking practice.

In [ ]:
# Features are used as created in Step 3.
print("Target column: 'default_event'")

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: always predict the majority class ---
baseline_clf = DummyClassifier(strategy='most_frequent', random_state=RANDOM_SEED)
baseline_clf.fit(X_train_scaled, y_train)
baseline_pred = baseline_clf.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline accuracy (majority-class): {baseline_acc:.3f}")
print(f"Any useful model must beat this {baseline_acc:.3f} floor.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

We train the model to predict the default event, but our real interest is the predicted probability.

In [ ]:
X = df.drop('default_event', axis=1)
y = df['default_event']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# Extracting PD (Probability of Default)
pd_estimates = model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(pd_estimates * 100, bins=50, color='#E76F51', edgecolor='#264653')
plt.title('Distribution of Estimated Probability of Default (PD)')
plt.xlabel('Probability of Default (%)')
plt.ylabel('Number of Loans')
plt.axvline(np.mean(pd_estimates)*100, color='#2A9D8F', linestyle='--', label='Portfolio Mean PD')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Histogram titled "Distribution of Estimated Probability of Default (PD)" with Probability of Default (%) on the x-axis and Number of Loans on the y-axis. The chart highlights distributional differences between groups that inform the modelling approach.

In [ ]:
test_results = pd.DataFrame({'actual': y_test, 'estimated_pd': pd_estimates})
test_results['pd_bucket'] = pd.qcut(test_results['estimated_pd'], 10)

calibration = test_results.groupby('pd_bucket').agg({'actual': 'mean', 'estimated_pd': 'mean'})

plt.figure(figsize=(6, 6))
plt.plot(calibration['estimated_pd'], calibration['actual'], 'o-', label='Model Calibration')
plt.plot([0, 0.4], [0, 0.4], '--k', label='Perfect Calibration')
plt.xlabel('Estimated PD')
plt.ylabel('Actual Default Rate')
plt.title('PD Calibration Curve')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Line chart titled "PD Calibration Curve" with Estimated PD on the x-axis and Actual Default Rate on the y-axis. The chart shows temporal trends and the alignment between predicted and observed values.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

**Summary:**
- We have successfully estimated the PD for each loan in the test set.
- The **Calibration Curve** shows that our model's estimates are close to the ground truth (the diagonal line).
- High LTV and DTI ratios significantly increase the PD estimate.

**Banking Context:** This PD output is used by banks for **IFRS 9 / CECL** accounting standards, where they must estimate future losses and set aside money accordingly.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: score representative cases from the test set ---
print("Operational demonstration — model decision support:\n")
proba = model.predict_proba(X_test)[:, 1]
low_idx  = int(np.argmin(proba))
high_idx = int(np.argmax(proba))
mid_idx  = int(np.argsort(proba)[len(proba) // 2])

for label, idx in [("Low-risk", low_idx), ("Medium-risk", mid_idx), ("High-risk", high_idx)]:
    row = X_test.iloc[idx]
    prob = proba[idx]
    print(f"{label} case  predicted probability: {prob:.1%}")
    print(f"  Features: {dict(row.round(2))}")
    print()

print("A decision-maker would combine these scores with business rules and domain judgement.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end credit scoring and risk assessment workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Credit models can perpetuate historical discrimination if trained on biased lending data.
- Proxy variables such as zip code or employment type may correlate with protected characteristics.
- Applicants deserve transparent explanations of adverse credit decisions under fair lending regulations.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in credit scoring and risk assessment?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.